### Import & Config

In [8]:
import requests
import json
import time
import os
import pandas as pd
from dotenv import load_dotenv

# Load API credentials from .env file
load_dotenv()
API_KEY = os.getenv("RAINFOREST_API_KEY")

# API configuration
BASE_URL = "https://api.rainforestapi.com/request"
OUTPUT_DIR = "../data/api_raw"
os.makedirs(OUTPUT_DIR, exist_ok=True)

### Load ASINs

In [3]:
with open('../data/api_raw/top_asins.json', 'r') as f:
    asins = json.load(f)

print(f"ASINs to query: {len(asins)}")
asins

ASINs to query: 32


['B0D8W1YVBX',
 'B08KT2Z93D',
 'B09541P9WH',
 'B00AHAWWO0',
 'B0DBF65JYY',
 'B0DPHQRLJC',
 'B0BK2SC18T',
 'B091NJQ29P',
 'B0113UZJE2',
 'B0CJ1B6D6S',
 'B07PZF3QS3',
 'B0CTJGJL2T',
 'B0F2TB2MMP',
 'B07HRCDDL1',
 'B0C3QZ7SNF',
 'B0CQMRKRV5',
 'B0DGHMNQ5Z',
 'B0FC5FJZ9Z',
 'B0FC6S2R7K',
 'B0FLXY2CBC',
 'B0B72DBKVF',
 'B01FWAZEIU',
 'B0F43VY6TC',
 'B09B2SBHQK',
 'B07HJSHT8P',
 'B088H5DWV3',
 'B0009KF59W',
 'B0CG2LW9RN',
 'B08JGNRJ8P',
 'B08P286Q26',
 'B0C9BLDRYS',
 'B09GNQD678']

### This is the request test with 1 product

In [ ]:
# try 1 request for product
params = {
    'api_key'       : API_KEY,
    'type'          :'product',
    'asin'          : asins[0],
    'amazon_domain' :'amazon.com'
}

response = requests.get(BASE_URL, params=params)
print("Status:", response.status_code)


if response.status_code == 200:
    data = response.json()
    product = data.get('product', {})
            
    # Categories
    categories = product.get('categories', [])

    # Buybox
    buybox = product.get('buybox_winner', {})
    fulfillment = buybox.get('fulfillment', {})
    seller = fulfillment.get('third_party_seller', {})

    # Best Sellers Rank
    bsr = product.get('bestsellers_rank', [])

    brand      = product.get('brand')
    subcategory  = categories[-1].get('name') if categories else None
    image      = product.get('main_image', {}).get('link')
    is_fba     = fulfillment.get('is_fulfilled_by_amazon')
    bsr_main     = bsr[0].get('rank') if len(bsr) > 0 else None
    bsr_sub      = bsr[1].get('rank') if len(bsr) > 1 else None
    seller_name = seller.get('name')
    print("\n--- First product ---")
    print("ASIN:    ", asins[0])
    print("brand:   ", brand)
    print('seller name', seller_name)
    print("is_fba:  ",is_fba)
    print("image:   ", image)
    print("subcategory:", subcategory)
    print("bsr_main: ",bsr_main)
    print('bsr_sub', bsr_sub)
        
else:
    print(f"Request failed — status {response.status_code}")
    print(response.text)

Status: 200
Products retrieved: 51

--- First product ---
ASIN:     B0D8W1YVBX
brand:    EQQUALBERRY
is_fba:   True
bsr_main:  22
seller_id:    None
image:    https://m.media-amazon.com/images/I/614vlOIc8YL.jpg
subcategory: Serums
bsr_sub 2
bsr_category Beauty & Personal Care


### Append the data to rows list

In [ ]:
rows = []

for asin in asins:
    
    params = {
        'api_key'       : API_KEY,
        'type'          :'product',
        'asin'          : asin,
        'amazon_domain' :'amazon.com'
    }
    
    response = requests.get(BASE_URL, params=params)

    if response.status_code != 200:
        print(f"Error {asin} - {response.status_code}")
        continue

    data = response.json()
    product = data.get('product', {})
            
    # Categories
    categories = product.get('categories', [])

    # Buybox
    buybox = product.get('buybox_winner', {})
    fulfillment = buybox.get('fulfillment', {})
    seller = fulfillment.get('third_party_seller', {})

    # Best Sellers Rank
    bsr = product.get('bestsellers_rank', [])

    rows.append({
        'asin':        asin,
        'brand':       product.get('brand'),
        'subcategory': categories[-1].get('name') if categories else None,
        'image':       product.get('main_image', {}).get('link'),
        'is_fba':      fulfillment.get('is_fulfilled_by_amazon'),
        'bsr_main':    bsr[0].get('rank') if len(bsr) > 0 else None,
        'bsr_sub':     bsr[1].get('rank') if len(bsr) > 1 else None,
        'seller_name': seller.get('name')
    })
    print(f"Succeed ({asin})")
    time.sleep(0.5)
            



Succeed (B0D8W1YVBX)
Succeed (B08KT2Z93D)
Succeed (B09541P9WH)
Succeed (B00AHAWWO0)
Succeed (B0DBF65JYY)
Succeed (B0DPHQRLJC)
Succeed (B0BK2SC18T)
Succeed (B091NJQ29P)
Succeed (B0113UZJE2)
Succeed (B0CJ1B6D6S)
Succeed (B07PZF3QS3)
Succeed (B0CTJGJL2T)
Succeed (B0F2TB2MMP)
Succeed (B07HRCDDL1)
Succeed (B0C3QZ7SNF)
Succeed (B0CQMRKRV5)
Succeed (B0DGHMNQ5Z)
Succeed (B0FC5FJZ9Z)
Succeed (B0FC6S2R7K)
Succeed (B0FLXY2CBC)
Succeed (B0B72DBKVF)
Succeed (B01FWAZEIU)
Succeed (B0F43VY6TC)
Succeed (B09B2SBHQK)
Succeed (B07HJSHT8P)
Succeed (B088H5DWV3)
Succeed (B0009KF59W)
Succeed (B0CG2LW9RN)
Succeed (B08JGNRJ8P)
Succeed (B08P286Q26)
Succeed (B0C9BLDRYS)
Succeed (B09GNQD678)


NameError: name 'pd' is not defined

### Create DataFrame with the rows created and appended

In [ ]:
df_products = pd.DataFrame(rows)
df_products.head(15)

,asin,brand,subcategory,image,is_fba,bsr_main,bsr_sub,seller_name
0,B0D8W1YVBX,EQQUALBERRY,Serums,https://m.media-amazon.com/images/I/614vlOIc8Y...,True,22.0,2.0,boosterskorea
1,B08KT2Z93D,eos,Lotions,https://m.media-amazon.com/images/I/51lP01--ej...,True,2.0,1.0,None
2,B09541P9WH,Amazon Basics,Cotton Swabs,https://m.media-amazon.com/images/I/612HeyYXOn...,None,41.0,2.0,None
3,B00AHAWWO0,Crest,Strips,https://m.media-amazon.com/images/I/715rFhZpV0...,True,121.0,1.0,None
4,B0DBF65JYY,medicube,Serums,https://m.media-amazon.com/images/I/615DOoCI6x...,True,49.0,4.0,Medicube & Aprilskin
5,B0DPHQRLJC,eos,Body Washes,https://m.media-amazon.com/images/I/51bThha-95...,True,12.0,1.0,None
6,B0BK2SC18T,GuruNanda,Strips,https://m.media-amazon.com/images/I/71EybBZ-jp...,True,139.0,2.0,GuruNanda LLC
7,B091NJQ29P,Good Molecules,Gels,https://m.media-amazon.com/images/I/71QsFtfDUC...,True,116.0,1.0,Good Molecules US
8,B0113UZJE2,‎Etekcity,Digital Scales,https://m.media-amazon.com/images/I/91YrLTBnMc...,True,3.0,1.0,None
9,B0CJ1B6D6S,Scrub Daddy,Sponges,https://m.media-amazon.com/images/I/71kiOOs1MG...,True,311.0,3.0,Front Row Group


In [12]:
df_products.to_csv('../data/processed/products_enriched.csv', index=False)
print(f"Saved: {df_products.shape}")

Saved: (32, 8)


Merge the DataFrames with the new info about the product by ASIN

In [ ]:
df_enriched = df.merge(df_products, on='asin', how='left')